In [1]:
import math

from Util import Problem, solution


class P004(Problem):
    number = 4
    title = "Largest Palindrome Product"
    description = (
        "<p>A palindromic number reads the same both ways. " +
        "The largest palindrome made from the product of two $2$-digit numbers" +
        "is $9009 = 91 \\times 99$.</p>" +
        "<p>Find the largest palindrome made from the product of two $3$-digit numbers.</p>"
    )

In [2]:
p = P004()
p.describe()

## Problem 4: Largest Palindrome Product

<p>A palindromic number reads the same both ways. The largest palindrome made from the product of two $2$-digit numbersis $9009 = 91 \times 99$.</p><p>Find the largest palindrome made from the product of two $3$-digit numbers.</p>

### Solution notes
First we cycle through all 3 digit numbers from highest to lowest, multiply them together and see if the result is a palindrome.

In [3]:
@solution(P004, first=True, make_fast=True)
def brute_force():
    def is_palindrome(number):
        number_text = str(number)
        for digit in range( 0, (len(number_text))//2):
            if number_text[digit] != number_text[-1 - digit]:
                return False
        return True
    largest_palindrome = 0
    for i in range(1000, 100, -1):
        for j in range(1000, 100, -1):
            product = i * j
            if is_palindrome(product):
                if product > largest_palindrome:
                    largest_palindrome = product
    return largest_palindrome

In [4]:
p.test_all(repeats=1)

906609 found in 91.235400 ms by brute_force (first)


We can optimise the method by cycling through fewer options (once we find a solution, we never need to search lower than the value of the lower factor for either factor). We also check if the product is smaller than the current result, if so, we need not perform any additional checks.

In [5]:
@solution(P004, make_fast=True)
def optimised_brute_force():
    def is_palindrome(number):
        number_text = str(number)
        for digit in range( 0, (len(number_text))//2):
            if number_text[digit] != number_text[-1 - digit]:
                return False
        return True
    largest_palindrome = 0
    min_j = 100
    for i in range(1000, min_j, -1):
        for j in range(1000, min_j, -1):
            product = i * j
            if product < largest_palindrome:
                continue
            if is_palindrome(product):
                if product > largest_palindrome:
                    min_j = j
                    largest_palindrome = product
    return largest_palindrome

In [6]:
p.test_all(repeats=1)

906609 found in 94.144600 ms by brute_force (first)
906609 found in 2.504900 ms by optimised_brute_force


Instead of checking the palindromes as strings, it is computationally less expensive to check them mathematically. I do this by calculating "palindrome identities" as I call them. For a 5-digit number, for example, these are $[10001, 1010, 100]$. With the definition of palindrome used here, a palindrome is always a multiple of the identities of that many digits.

In [7]:
@solution(P004, best= True, make_fast=True)
def better_palindrome_check_brute_force():
    def is_palindrome(number):
        # This is really the digits - 1, but as we are using this for powers of 10, we would
        # otherwise have to add it here and subtract it again later
        digits_in_number = math.floor(math.log10(number))
        l = digits_in_number
        while l >= digits_in_number / 2:
            if number % (10 ** (digits_in_number - l)) != 0:
                return False
            if number == 0:
                return True
            if l == math.floor(math.log10(abs(number))):
                palindrome_identity = (
                    10 ** l + 10 ** (digits_in_number - l) if l != digits_in_number - l else
                    10 ** l
                )
                first_digit = number // 10 ** l
                number -= palindrome_identity * first_digit
            l -= 1
        return number == 0
    largest_palindrome = 0
    min_j = 100
    for i in range(1000, min_j, -1):
        for j in range(1000, min_j, -1):
            product = i * j
            if product < largest_palindrome:
                continue
            if is_palindrome(product):
                if product > largest_palindrome:
                    min_j = j
                    largest_palindrome = product
    return largest_palindrome

In [8]:
p.test_all(repeats=100)

906609 found in 0.274960 ms by better_palindrome_check_brute_force (best)
906609 found in 89.483785 ms by brute_force (first)
906609 found in 0.985365 ms by optimised_brute_force
